# Reproducing Pakhomov et al. (2023) from dandiset 000686

Paper. Pakhomov, Gudvangen, Mangalanathan, Kondratiev, Redondo & Semenov (2023), "Next generation
CANCAN focusing for remote stimulation by nanosecond electric pulses", Bioelectrochemistry 152,
108437, [doi:10.1016/j.bioelechem.2023.108437](https://doi.org/10.1016/j.bioelechem.2023.108437).
The claims file (`claims/000686/10.1016-j.bioelechem.2023.108437.yaml`) was written from the accepted
author manuscript in PubMed Central (PMC10247516), Methods, Results, Discussion and legends; the paper
has no supplement and links no code.

Dataset. [DANDI:000686](https://dandiarchive.org/dandiset/000686/0.240515.1909), version
0.240515.1909, streamed from the archive. It is the only published version (the draft has the same
18 assets and was last modified a second after it was published), and the paper cites no version, so
the most recent published version is pinned. The dandiset holds 18 NWB files, one per dish, each a
single stitched YO-PRO-1 image of a cell monolayer inside the quadrupole array after treatment: the
four groups of Fig. 5 (40 Packets and 40 Trains from e1, n = 4 each; 1P x 40 and 1T x 40 with
rotation, n = 5 each), all Packet 2 or its train at 6.4 kV. No ROI measurements are deposited: the
profiles are measured here from the images, with the ROI layout of Methods 2.4.

| Claim | Figure | Status here |
|---|---|---|
| fig3-field-simulation | 3 | blocked: data_missing (simulated field maps not deposited) |
| fig4-sham-no-uptake, fig4-unipolar-uptake, fig4-bipolar-weaker, fig4-8phase-cancellation | 4 | blocked: data_missing (single-cell time-lapse not deposited) |
| fig5a-trains-stronger | 5A | tested |
| fig5a-packet-center-enhanced | 5A | tested |
| fig5b-rotation-center-peak | 5B | tested |
| fig5c-normalized-contrast | 5C | tested |
| fig6a, fig6b, fig6-void-between, fig6-peak-narrower | 6 | blocked: other (data in DANDI:000875, not in this dandiset) |
| fig7a-amplitude-scales-uniformly, fig7-amplitude-no-effect-on-distribution | 7 | blocked: other (data in DANDI:001025) |
| fig8-center-equal-packets, fig8a-periphery-packets-1-3-higher, fig8b-normalized-center | 8 | blocked: other (data in DANDI:001027) |
| fig9a-gap-reduces-periphery-more, fig9b-gap-boosts-targeting | 9 | blocked: other (data in DANDI:001028) |

How the profiles are measured (helper `cancan.py`, which cites the source of every parameter).
Each image is streamed and reduced to 8 x 8 pixel block means. The four electrode footprints (the
"dark circles" of the file description) are found automatically. Along each diagonal, square ROIs
run from the array centre (the mean of the four footprint centres) to each electrode: 56 per half
diagonal as in Methods 2.4 (224 ROIs along both diagonals), plus one on the centre, ending at
3.65/4.2 of the distance from the centre to the footprint's inner edge (Methods 2.4, +/- 3.65 mm; Results
3.5, 8.4-mm diagonal). The readings O1 to O6 of the claims file are run as it names them: the centre
as the centre ROI or a LOESS fit at 0 (spans 0.1, 0.2, 0.3), the periphery as the outermost ROI at
both ends or at the e1 end, raw values or values minus a background (the median inside the
footprints), normalization per dish or of the mean, and paired and unpaired t-tests where the
claims file asks for both. Tests are two-sided Student's t-tests (Methods 2.4) over dishes.

In [ ]:
import collections
import re
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import cancan

print("Dandiset", cancan.DANDISET_ID, "version", cancan.DANDISET_VERSION)


def load(r):
    """Stream one file, reduce it to 8 x 8 block means, find the electrodes and read the profiles."""
    img, meta = cancan.load_image(r["url"])
    el = cancan.find_electrodes(img)
    h, w = (img.shape[0] // 8) * 8, (img.shape[1] // 8) * 8
    coarse = np.nanmean(img[:h, :w].reshape(h // 8, 8, w // 8, 8), axis=(1, 3))
    return dict(**r, meta=meta, el=el, prof=cancan.diagonal_profiles(img, el),
                prof3=cancan.diagonal_profiles(img, el, width=3), bg=cancan.footprint_background(img, el),
                coarse=coarse)


import warnings
warnings.filterwarnings("ignore", "Mean of empty slice")   # blocks outside the stitched tiles
with ThreadPoolExecutor(3) as ex:
    D = list(ex.map(load, cancan.assets()))
print("loaded", len(D), "images")

## Data check

Counts by group and date, the metadata of the image arrays, the geometry found in each image, and
a search for files deposited twice. Distances in mm assume 0.65 um per pixel (see the helper); no
measure depends on it.

In [ ]:
import collections
groups = ["40 Packets", "40 Trains", "1P x 40", "1T x 40"]
print("Dandiset", cancan.DANDISET_ID, "version", cancan.DANDISET_VERSION, "-", len(D), "NWB files")
print("Files per group:", dict(collections.Counter(d["group"] for d in D)))
print("Sessions by date:", dict(collections.Counter((d["group"], d["meta"]["session_start"][:10]) for d in D)))
m0 = D[0]["meta"]
print("\nImage array: shapes", {d["meta"]["shape"] for d in D}, "dtype", {d["meta"]["dtype"] for d in D},
      "filters", {tuple(d["meta"]["filters"]) for d in D})
print("unit", {d["meta"]["unit"] for d in D}, "resolution", {d["meta"]["resolution"] for d in D},
      "conversion", {d["meta"]["conversion"] for d in D}, "(no pixel size is given)")
print("species", {d["meta"]["species"] for d in D}, "(NCBITaxon_9913 is Bos taurus; the paper's cells are CHO-K1, hamster)")
print("Image description:", m0["image_description"])
print("\nTreatment named in each session description (pulses per packet or train, first-pulse kV, cycles):")
for g in groups:
    descs = {re.sub(r"\s+", " ", d["meta"]["session_description"]) for d in D if d["group"] == g}
    for s in descs:
        print(f"  {g}: {len(descs)} distinct;", re.findall(r"(?:packets|trains) of \d+ pulses|[\d.]+ kV|total of \d+ \w+", s))
rows = []
for d in D:
    el, m = d["el"], d["meta"]
    px_mm = cancan.BLOCK * cancan.UM_PER_PIXEL / 1000
    rows.append(dict(group=d["group"], dish=d["dish"], start=m["session_start"][:16],
                     diag_centres_mm=el["diagonal"] * px_mm, footprint_diam_mm=2 * el["radius"] * px_mm,
                     diag_inner_edges_mm=(el["diagonal"] - 2 * el["radius"]) * px_mm,
                     square_err=el["square_error"], roi_end_mm=d["prof"]["mm"][:, -1].mean(),
                     roi_step_um=1000 * d["prof"]["mm"][:, 1].mean(), background=d["bg"],
                     median=np.nanmedian(d["coarse"]), max=m["max"], frac_ge_65000=m["frac_ge_65000"],
                     frac_no_data=m["frac_zero"]))
T = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_columns", 30, "display.precision", 3):
    print(T.to_string(index=False))
print("\nRanges: centre-to-centre diagonal %.2f to %.2f mm, footprint diameter %.2f to %.2f mm, inner-edge diagonal "
      "%.2f to %.2f mm (at %.2f um per pixel)" % (T.diag_centres_mm.min(), T.diag_centres_mm.max(), T.footprint_diam_mm.min(),
      T.footprint_diam_mm.max(), T.diag_inner_edges_mm.min(), T.diag_inner_edges_mm.max(), cancan.UM_PER_PIXEL))
print("Tile geometry: 22323 px from 12 tiles is %.0f px per tile step, consistent with 2048-px ORCA-Flash4 tiles "
      "at 10%% overlap (6.5 um / 10x = 0.65 um per pixel)." % ((22323 - 2048) / 11))
print("Profile values over all dishes and ROIs: %.0f to %.0f (raw block means, a.u.)" % (
      min(np.nanmin(d["prof"]["halves"]) for d in D), max(np.nanmax(d["prof"]["halves"]) for d in D)))
print("NaN ROIs:", sum(int(np.isnan(d["prof"]["halves"]).sum()) for d in D))

# Recordings deposited twice: correlation of the coarse images between every pair of files
C = np.array([d["coarse"].ravel() for d in D], dtype=float)
C = np.nan_to_num(C)
R = np.corrcoef(C)
iu = np.triu_indices(len(D), 1)
o = np.argsort(R[iu])[::-1][:3]
print("\nHighest image correlations between two files (1 would be a duplicate):")
for k in o:
    i, j = iu[0][k], iu[1][k]
    print(f"  {D[i]['group']} {D[i]['dish']} / {D[j]['group']} {D[j]['dish']}: r = {R[i, j]:.3f}")
cent = np.array([d["el"]["centres"].ravel() for d in D])
dmin = min(np.abs(cent[i] - cent[j]).max() for i, j in zip(*iu))
print("Smallest largest-difference of footprint positions between two files: %.0f px" % dmin)
print("Subject ids are unique:", len({d['meta']['subject_id'] for d in D}) == len(D))
print("Timing: each file holds one image taken after treatment; there is no time series or trial table, "
      "so there is no clock to check.")

### Images

Each dish's image at 1/64 of the resolution, with the four footprints found and the ROIs of both
diagonals (every 4th ROI drawn). Brightness is scaled to each image's 99.5th percentile.

In [ ]:
fig, axs = plt.subplots(3, 6, figsize=(24, 12.5))
for a, d in zip(axs.flat, D):
    x = d["coarse"]
    a.imshow(x, cmap="gray", vmax=np.nanpercentile(x, 99.5))
    for i in range(4):
        pts, _ = cancan.roi_positions(d["el"], i)
        a.plot(pts[::4, 1] / 8, pts[::4, 0] / 8, ".", ms=1.5, color="C1" if i in (1, 3) else "C2")
    c = d["el"]["centres"] / 8
    a.plot(c[:, 1], c[:, 0], "o", mfc="none", color="C3", ms=12)
    a.set_title(f"{d['group']} {d['dish']}", fontsize=10); a.axis("off")
plt.tight_layout(); plt.show()

### Which electrode is e1

The files do not say which footprint is e1. Fig. 5A and Results 3.2 need it (the profile runs from e1 to e3).
Two readings (i) the electrode with the brightest outermost ROIs in the non-rotated dishes; (ii)
the top-right electrode of the image, the mirror image of the top-left electrode energized in Fig. 3
(an inverted microscope sees the dish from below). Reading (i) uses the uptake near the electrode
and so is partly circular for the train profile, which is why reading (ii) is shown with it.

In [ ]:
# Which electrode is e1. Reading (i): e1 is the electrode with the brightest outer ROIs in the
# non-rotated dishes (for trains, Results 3.2 says uptake follows the field, strongest near e1).
# Reading (ii): e1 is the top-right electrode of the image, the mirror of the top-left electrode
# energized in Fig. 3, as an inverted microscope sees the dish from below.
names = ["top left", "top right", "bottom right", "bottom left"]
print("Mean of the 3 outermost ROIs at each electrode (image clockwise from top left):")
bright = []
for d in D:
    if d["rotated"]:
        continue
    v = d["prof"]["halves"][:, -3:].mean(1)
    bright.append(int(np.argmax(v)))
    print(f"  {d['group']:10s} {d['dish']:9s}", np.round(v).astype(int), "brightest:", names[bright[-1]])
print("Brightest electrode over the 8 non-rotated dishes:", dict(collections.Counter(names[b] for b in bright)))
E1, E3 = 1, 3          # top right and bottom left; both readings agree when every dish is top right
E2, E4 = 2, 0          # the other diagonal; E2 next to E1 clockwise in the image (a labelling choice)
print("e1 = top right under both readings:", all(b == E1 for b in bright))

### Mean profiles

Fig. 5A (one diagonal, e3 at left, e1 at right) and Fig. 5B (both diagonals averaged by signed
distance). Mean +/- s.e. over dishes.

In [ ]:
def group(g):
    return [d for d in D if d["group"] == g]

def prof(d, bg=False, width=1):
    """Signed profile along e1-e3 (index 0 at e3, 56 at centre, 112 at e1)."""
    h = d["prof"]["halves"] if width == 1 else d["prof3"]["halves"]
    return cancan.signed_profile(h, E1, E3) - (d["bg"] if bg else 0.0)

def prof_both(d, bg=False, width=1):
    """Profile averaged over both diagonals by signed distance (Fig. 5B legend); the positive end
    averages the e1 and e2 ends."""
    h = d["prof"]["halves"] if width == 1 else d["prof3"]["halves"]
    return (cancan.signed_profile(h, E1, E3) + cancan.signed_profile(h, E2, E4)) / 2 - (d["bg"] if bg else 0.0)

O1 = ["roi"] + [f"loess{s}" for s in cancan.LOESS_SPANS]
O4 = {"raw": False, "bg": True}
x_mm = np.concatenate([-D[0]["prof"]["mm"][E3][::-1], D[0]["prof"]["mm"][E1][1:]])
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for g, c in [("40 Packets", "C0"), ("40 Trains", "C3")]:
    Y = np.array([prof(d) for d in group(g)])
    xs = np.mean([np.concatenate([-d["prof"]["mm"][E3][::-1], d["prof"]["mm"][E1][1:]]) for d in group(g)], 0)
    ax[0].errorbar(xs, Y.mean(0), Y.std(0, ddof=1) / np.sqrt(len(Y)), color=c, lw=1, label=f"{g} (n={len(Y)})")
    ax[0].plot(xs, cancan.loess(Y.mean(0), 0.2), color="k", lw=0.8)
    print(f"{g}: mean profile range {Y.mean(0).min():.0f} to {Y.mean(0).max():.0f} a.u.")
ax[0].set(xlabel="distance from centre toward e1 (mm, at 0.65 um/px)", ylabel="YP intensity (a.u.)", title="Fig. 5A: e3 (left) to e1 (right)")
ax[0].legend()
for g, c in [("1P x 40", "C0"), ("1T x 40", "C3")]:
    Y = np.array([prof_both(d) for d in group(g)])
    ax[1].errorbar(x_mm, Y.mean(0), Y.std(0, ddof=1) / np.sqrt(len(Y)), color=c, lw=1, label=f"{g} (n={len(Y)})")
    ax[1].plot(x_mm, cancan.loess(Y.mean(0), 0.2), color="k", lw=0.8)
    print(f"{g}: mean profile range {Y.mean(0).min():.0f} to {Y.mean(0).max():.0f} a.u.")
ax[1].set(xlabel="distance from centre (mm)", title="Fig. 5B: averaged over both diagonals")
ax[1].legend(); plt.tight_layout(); plt.show()
print("All points are inside the axes (autoscaled). Black lines: LOESS, span 0.2, of the mean profile.")

## Claims tested

### fig5a-trains-stronger

> Without rotation, the control of 40 unipolar trains from e1 causes much stronger electroporation throughout the array than 40 Packet 2 packets started from e1. The uptake after trains follows the field strength distribution and falls with distance from e1.

Criterion (claims file): trains above packets at the e1 end (outermost ROI), the centre (O1) and the e3 end, each by an unpaired two-sided Student's t-test at p < 0.05, and the train e1-end value above the train centre by the same test; under O1 (a, b with spans 0.1, 0.2, 0.3) and O4 (raw, minus footprint background).

In [ ]:
P, Tr = group("40 Packets"), group("40 Trains")
rows = []
for o1 in O1:
    for o4, bg in O4.items():
        vals = {}
        for name, G in [("P", P), ("T", Tr)]:
            Y = [prof(d, bg) for d in G]
            vals[name] = dict(e1=np.array([y[-1] for y in Y]), centre=np.array([cancan.centre_value(y, o1) for y in Y]),
                              e3=np.array([y[0] for y in Y]))
        r = dict(O1=o1, O4=o4)
        ok = True
        for place in ["e1", "centre", "e3"]:
            t, p = cancan.ttest(vals["T"][place], vals["P"][place])
            r[f"T/P {place}"] = f"{vals['T'][place].mean():.0f}/{vals['P'][place].mean():.0f} p={p:.3g}"
            ok &= (t > 0) and (p < 0.05)
        t, p = cancan.ttest(vals["T"]["e1"], vals["T"]["centre"])
        tp, pp = cancan.ttest(vals["T"]["e1"], vals["T"]["centre"], paired=True)
        r["T e1>centre unpaired"] = f"t={t:.2f} p={p:.3g}"
        r["(paired variant)"] = f"t={tp:.2f} p={pp:.3g}"
        r["e3 bar gap (s.e.)"] = round(cancan.errorbar_gap(vals["T"]["e3"], vals["P"]["e3"])[0], 2)
        ok &= (t > 0) and (p < 0.05)
        r["holds"] = ok
        rows.append(r)
R5a = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(R5a.to_string(index=False))
print("\nReadings where every condition holds:", int(R5a.holds.sum()), "of", len(R5a))
print("Per dish, raw ROI values (e1 end, centre, e3 end):")
for name, G in [("40 Packets", P), ("40 Trains", Tr)]:
    for d in G:
        y = prof(d)
        print(f"  {name:10s} {d['dish']:9s} {y[-1]:8.0f} {y[56]:8.0f} {y[0]:8.0f}")
# Train profile shape: ROI means of the group mean profile from the e3 end to the e1 end, every 8 ROIs
Ym = np.mean([prof(d) for d in Tr], 0)
print("Train mean profile every 8th ROI from e3 (-) to e1 (+):", np.round(Ym[::8]).astype(int))
Ym3 = np.mean([prof(d, width=3) for d in Tr], 0); Pm3 = np.mean([prof(d, width=3) for d in P], 0)
print("Variant, ROIs 3 steps wide: trains e1/centre/e3 %.0f/%.0f/%.0f, packets %.0f/%.0f/%.0f"
      % (Ym3[-1], Ym3[56], Ym3[0], Pm3[-1], Pm3[56], Pm3[0]))

**Result: partially reproduced.** The 40 trains exceed the 40 packets in the centre under all 8 readings of O1 and O4 (p = 0.00355 to 0.0443), and the train e1-end value exceeds the train centre (unpaired p = 0.0244 to 0.034). Near e1 the trains are higher in the mean (13565 against 6347 a.u.) but not significantly with one-step ROIs (p = 0.059; with ROIs three steps wide p = 0.0042). At the e3 end the two groups do not differ under any reading (1099 against 1285 a.u., p = 0.52; 799 against 1053 after background, p = 0.35). The criterion therefore fails under every reading. Sampled every 8th ROI, the train mean profile is 13565 to 17262 a.u. over the 2 mm nearest e1 and 831 to 1099 a.u. near e3, falling with distance from e1 as the field does; the stronger uptake after trains holds in the centre and, in the mean, near e1, not at the far end of the diagonal.

### fig5a-packet-center-enhanced

> Without rotation, Packet 2 enhances electroporation in the center of the array despite the weakening of the field with distance from e1. Uptake near e1 still stays stronger than in the center.

Criterion (claims file): on the 40-packet e1-e3 profile, the centre (O1) above the lowest value between the centre and the e1 end, and the e1-end value above the centre, each by a two-sided t-test at p < 0.05 over dishes, paired and unpaired (O6), under all readings of O1 and O4. The trough is the lowest point of the group's mean profile (ROI values or the mean of the dishes' LOESS fits, as O1) strictly between the centre and the e1 end; each dish's value at that ROI enters the test.

In [ ]:
rows = []
for o1 in O1:
    for o4, bg in O4.items():
        Y = np.array([prof(d, bg) for d in P])
        S = np.array([cancan.smooth(y, o1) for y in Y])
        between = np.arange(57, 112)            # strictly between the centre (56) and the e1 end (112)
        k = between[np.argmin(S.mean(0)[between])]
        centre, trough, e1 = S[:, 56], S[:, k], Y[:, -1]
        for paired in (False, True):
            t1, p1 = cancan.ttest(centre, trough, paired)
            t2, p2 = cancan.ttest(e1, centre, paired)
            rows.append(dict(O1=o1, O4=o4, test="paired" if paired else "unpaired",
                             trough_mm=round(float(x_mm[k]), 2), centre=round(centre.mean()), trough=round(trough.mean()),
                             e1_end=round(e1.mean()), **{"centre>trough": f"t={t1:.2f} p={p1:.3g}",
                             "e1>centre": f"t={t2:.2f} p={p2:.3g}"},
                             holds_centre=(t1 > 0) and (p1 < 0.05), holds_e1=(t2 > 0) and (p2 < 0.05),
                             gap_centre_trough=round(cancan.errorbar_gap(centre, trough)[0], 2),
                             gap_e1_centre=round(cancan.errorbar_gap(e1, centre)[0], 2)))
R5ap = pd.DataFrame(rows)
R5ap["holds"] = R5ap.holds_centre & R5ap.holds_e1
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(R5ap.to_string(index=False))
print("\nReadings where centre > trough holds:", int(R5ap.holds_centre.sum()), "of", len(R5ap),
      "; e1 end > centre holds:", int(R5ap.holds_e1.sum()), "; both:", int(R5ap.holds.sum()))
print("Per dish (raw ROI): centre, e1 end, ratio e1/centre")
for d in P:
    y = prof(d)
    print(f"  {d['dish']:9s} {y[56]:7.0f} {y[-1]:7.0f} {y[-1] / y[56]:5.2f}")
Y = np.array([prof(d) for d in P])
print("Packet mean profile every 8th ROI from e3 (-) to e1 (+):", np.round(Y.mean(0)[::8]).astype(int))

**Result: not reproduced.** The e1-end value of the 40-packet profile exceeds the centre in the mean (6347 against 2526 a.u., raw ROIs) and in all 4 dishes, but under no reading is the difference significant (p = 0.105 to 0.162, paired or unpaired), and by the paper's error-bar rule it is not either (gap 0.60 to 0.84 s.e.). The centre exceeds the trough between centre and e1 under 10 of 16 readings, every LOESS reading unpaired and spans 0.1 and 0.2 paired (p = 0.0043 to 0.0371), not with the centre ROI itself (p = 0.17 to 0.24) or span 0.3 paired (p = 0.068). As the criterion needs both comparisons and the second fails under every reading, the outcome is the same under all readings. With ROIs three steps wide the centre exceeds the trough (p = 0.001) and the e1 end still does not exceed the centre (p = 0.16).

### fig5b-rotation-center-peak

> With rotation (1P x 40), Packet 2 smears the effect at the perimeter and brings out a remote effect in the center. The same rotation of unipolar trains (1T x 40) leaves a void of electroporation in the center.

Criterion (claims file): profiles averaged across both diagonals; the 1P x 40 centre (O1) above its lowest value between centre and periphery (on either side, found on the mean profile as above), and the 1T x 40 centre below its most peripheral value (O2: both ends, or the e1 end), each by a two-sided t-test at p < 0.05 (O6: paired and unpaired both run), under all readings of O1, O2 and O4.

In [ ]:
RP, RT = group("1P x 40"), group("1T x 40")
rows = []
for o1 in O1:
    for o4, bg in O4.items():
        YP = np.array([prof_both(d, bg) for d in RP]); SP = np.array([cancan.smooth(y, o1) for y in YP])
        YT = np.array([prof_both(d, bg) for d in RT]); ST = np.array([cancan.smooth(y, o1) for y in YT])
        between = np.r_[1:56, 57:112]          # strictly between the centre and either end
        k = between[np.argmin(SP.mean(0)[between])]
        cP, trough = SP[:, 56], SP[:, k]
        cT = ST[:, 56]
        for o2 in ["both", "e1"]:
            perT = (YT[:, 0] + YT[:, -1]) / 2 if o2 == "both" else YT[:, -1]
            for paired in (False, True):
                t1, p1 = cancan.ttest(cP, trough, paired)
                t2, p2 = cancan.ttest(perT, cT, paired)
                rows.append(dict(O1=o1, O4=o4, O2=o2, test="paired" if paired else "unpaired",
                                 P_centre=round(cP.mean()), P_trough=round(trough.mean()), trough_mm=round(float(x_mm[k]), 2),
                                 **{"P centre>trough": f"t={t1:.2f} p={p1:.3g}"},
                                 T_centre=round(cT.mean()), T_periph=round(perT.mean()),
                                 **{"T periph>centre": f"t={t2:.2f} p={p2:.3g}"},
                                 holds_P=(t1 > 0) and (p1 < 0.05), holds_T=(t2 > 0) and (p2 < 0.05),
                                 T_centre_lowest=bool(np.argmin(ST.mean(0)) == 56),
                                 gap_P=round(cancan.errorbar_gap(cP, trough)[0], 2),
                                 gap_T=round(cancan.errorbar_gap(perT, cT)[0], 2)))
R5b = pd.DataFrame(rows)
R5b["holds"] = R5b.holds_P & R5b.holds_T
with pd.option_context("display.width", 260, "display.max_columns", 20):
    print(R5b.to_string(index=False))
print("\n1P x 40 centre > trough holds in", int(R5b.holds_P.sum()), "of", len(R5b), "readings; 1T x 40 periphery > centre in",
      int(R5b.holds_T.sum()), "; both in", int(R5b.holds.sum()))
print("1T x 40 centre is the lowest point of its mean profile (reported, not required):",
      dict(collections.Counter(zip(R5b.O1, R5b.T_centre_lowest))))
YT = np.array([prof_both(d) for d in RT])
print("1T x 40: mean profile minimum at %.2f mm (%.0f a.u.), centre %.0f" % (x_mm[np.argmin(YT.mean(0))], YT.mean(0).min(), YT.mean(0)[56]))
print("Per dish (raw ROI, averaged diagonals): centre, mean of both ends")
for name, G in [("1P x 40", RP), ("1T x 40", RT)]:
    for d in G:
        y = prof_both(d)
        print(f"  {name:8s} {d['dish']:9s} {y[56]:7.0f} {(y[0] + y[-1]) / 2:7.0f}")

**Result: reproduced, resting on two assumptions.** Under the LOESS readings of the centre (O1 b, all three spans), the 1P x 40 centre exceeds the lowest value of its profile in all 24 readings (2142 to 2761 against 926 to 1204 a.u., p = 0.00452 to 0.031), and the 1T x 40 centre is below its most peripheral value (5169 to 5770 against 10494 to 10943 a.u.) by the unpaired test in all 12 unpaired readings (p = 0.0191 to 0.0287), not by the paired test (p = 0.0507 to 0.0797). With the centre ROI itself (O1 a) the 1P x 40 peak is not significant (3550 against 1038 a.u., p = 0.073 unpaired, 0.125 paired; error-bar gap 0.97 s.e.), while the 1T x 40 comparison holds both ways (p = 0.0111 to 0.0362). The outcome takes the LOESS centre and the unpaired test (see the attempt record). With ROIs three steps wide and the centre ROI, both comparisons hold (p = 0.003 and 0.017). The 1T x 40 centre is the lowest point of its ROI mean profile (4766 a.u.) but not of its LOESS fits.

### fig5c-normalized-contrast

> Normalized to the most peripheral ROI (100%), rotated Packet 2 and rotated trains show sharply different spatial distributions. The packet's center is relatively high and the trains' center is relatively low.

Criterion (claims file): the normalized centre (O1, O2, O3, O4) of 1P x 40 above that of 1T x 40 by an unpaired two-sided t-test at p < 0.05, under all readings; the normalized values, and whether 1P x 40 is above 100%, are reported. O3 (a): each dish normalized to its own peripheral ROI, then averaged; (b): the group's mean centre over its mean periphery, tested on each dish's centre divided by its group's mean periphery.

In [ ]:
rows = []
norm_profiles = {}
for o1 in O1:
    for o4, bg in O4.items():
        for o2 in ["both", "e1"]:
            per, cen = {}, {}
            for name, G in [("P", RP), ("T", RT)]:
                Y = np.array([prof_both(d, bg) for d in G])
                cen[name] = np.array([cancan.centre_value(y, o1) for y in Y])
                per[name] = (Y[:, 0] + Y[:, -1]) / 2 if o2 == "both" else Y[:, -1]
                if o1 == "roi" and o2 == "both":
                    norm_profiles[(o4, name)] = 100 * Y / per[name][:, None]
            for o3 in ["a", "b"]:
                if o3 == "a":
                    nP, nT = 100 * cen["P"] / per["P"], 100 * cen["T"] / per["T"]
                    vP, vT = nP.mean(), nT.mean()
                else:
                    nP, nT = 100 * cen["P"] / per["P"].mean(), 100 * cen["T"] / per["T"].mean()
                    vP, vT = 100 * cen["P"].mean() / per["P"].mean(), 100 * cen["T"].mean() / per["T"].mean()
                t, p = cancan.ttest(nP, nT)
                rows.append(dict(O1=o1, O4=o4, O2=o2, O3=o3, P_norm_centre=round(vP, 1), T_norm_centre=round(vT, 1),
                                 diff_points=round(vP - vT, 1), t=round(t, 2), p=float(f"{p:.3g}"),
                                 holds=(t > 0) and (p < 0.05), P_above_100=vP > 100,
                                 gap=round(cancan.errorbar_gap(nP, nT)[0], 2)))
R5c = pd.DataFrame(rows)
with pd.option_context("display.width", 250, "display.max_columns", 20):
    print(R5c.to_string(index=False))
print("\nReadings where 1P x 40 > 1T x 40 at p < 0.05:", int(R5c.holds.sum()), "of", len(R5c),
      "; 1P x 40 normalized centre above 100% in", int(R5c.P_above_100.sum()))
print("Ranges: 1P x 40 %.0f to %.0f %%, 1T x 40 %.0f to %.0f %%; p from %.2g to %.2g" % (
      R5c.P_norm_centre.min(), R5c.P_norm_centre.max(), R5c.T_norm_centre.min(), R5c.T_norm_centre.max(), R5c.p.min(), R5c.p.max()))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for a, o4 in zip(ax, ["raw", "bg"]):
    for name, c in [("P", "C0"), ("T", "C3")]:
        Y = norm_profiles[(o4, name)]
        a.errorbar(x_mm, Y.mean(0), Y.std(0, ddof=1) / np.sqrt(len(Y)), color=c, lw=1,
                   label=("1P x 40" if name == "P" else "1T x 40") + f" (range {Y.mean(0).min():.0f} to {Y.mean(0).max():.0f} %)")
    a.axhline(100, color="k", lw=0.5); a.set(title=f"Fig. 5C, O4 {o4}, O3 (a), O2 both ends", xlabel="distance from centre (mm)", ylabel="% of most peripheral ROI")
    a.legend(fontsize=8)
plt.tight_layout(); plt.show()

**Result: reproduced, resting on two assumptions.** The 1P x 40 normalized centre is above 100% under all 32 readings (134 to 268%) and the 1T x 40 one below (42 to 63%). The difference is significant under 24 of 32 readings: 23 of the 24 LOESS readings (p = 0.00071 to 0.0429; the exception, span 0.3 with background, e1 end and per-dish normalization, has p = 0.0515), and 1 of the 8 centre-ROI readings (p = 0.0476 to 0.0571). The outcome takes the LOESS centre, as for fig5b-rotation-center-peak, and the periphery at both ends (O2 a), under which all 12 LOESS readings hold (p = 0.00071 to 0.0248). With ROIs three steps wide and the centre ROI, 175.2% against 57.8%, p = 0.0139.

### ROI size

The paper gives no ROI size. The main readings are repeated with ROIs three steps wide.

In [ ]:
# ROI size is not given in the paper. Variant: ROIs 3 steps wide (about 195 um across the diagonal,
# overlapping along it), main readings only (centre ROI, raw values, unpaired tests).
def tests_at(width):
    out = {}
    YP = np.array([prof(d, width=width) for d in P]); YT = np.array([prof(d, width=width) for d in Tr])
    out["5A trains>packets e1, centre, e3 (p)"] = [round(cancan.ttest(YT[:, i], YP[:, i])[1], 4) for i in (-1, 56, 0)]
    out["5A trains e1>centre (p)"] = round(cancan.ttest(YT[:, -1], YT[:, 56])[1], 4)
    k = 57 + np.argmin(YP.mean(0)[57:112])
    out["5A packets centre>trough, e1>centre (t, p)"] = [tuple(round(v, 3) for v in cancan.ttest(YP[:, 56], YP[:, k])),
                                                     tuple(round(v, 3) for v in cancan.ttest(YP[:, -1], YP[:, 56]))]
    QP = np.array([prof_both(d, width=width) for d in RP]); QT = np.array([prof_both(d, width=width) for d in RT])
    b = np.r_[1:56, 57:112]; k = b[np.argmin(QP.mean(0)[b])]
    perT = (QT[:, 0] + QT[:, -1]) / 2
    out["5B 1P centre>trough, 1T periphery>centre (t, p)"] = [tuple(round(v, 3) for v in cancan.ttest(QP[:, 56], QP[:, k])),
                                                        tuple(round(v, 3) for v in cancan.ttest(perT, QT[:, 56]))]
    nP = 100 * QP[:, 56] / ((QP[:, 0] + QP[:, -1]) / 2); nT = 100 * QT[:, 56] / perT
    out["5C normalized centre 1P, 1T (%), t, p"] = (round(float(nP.mean()), 1), round(float(nT.mean()), 1), *[round(v, 4) for v in cancan.ttest(nP, nT)])
    return out
for w in (1, 3):
    print(f"ROI width {w} step(s):")
    for key, v in tests_at(w).items():
        print(f"  {key}: {v}")

## Claims not tested

- fig3-field-simulation (Fig. 3, blocked, `data_missing`). The values are outputs of the paper's
  Sim4Life model. Neither this dandiset nor any other dandiset citing the paper holds field maps, and
  the paper names no deposit. The criterion forbids a model built here.
- fig4-sham-no-uptake, fig4-unipolar-uptake, fig4-bipolar-weaker, fig4-8phase-cancellation
  (Fig. 4, blocked, `data_missing`). They need the confocal time-lapse of single cells on coverslips
  (Methods 2.2). This dandiset holds only the monolayer images of Fig. 5, and none of the four other
  dandisets that cite the paper (000875, 001025, 001027, 001028) holds single-cell time-lapse
  data; a search of DANDI for the lab's other dandisets found none that cites the paper.
  fig4-8phase-cancellation depends on the other two Fig. 4 claims.
- Figs. 6 to 9 (eleven claims, blocked, `other`). The data are not in dandiset 000686. The
  paper's authors deposited them in DANDI:000875 (Fig. 6), 001025 (Fig. 7), 001027 (Fig. 8) and
  001028 (Fig. 9), each of which names this paper. The claims file gives these claims
  `data: dataset`, so outside data may not be used (protocol, author, step 4); a question asks a
  reader whether they should be `data: elsewhere` (`questions/000686.yaml`). The block is not
  `data_missing`, as the data are public.

## Summary

| Claim | Outcome |
|---|---|
| fig5a-trains-stronger | partially reproduced: trains exceed packets in the centre, not at the e3 end, and near e1 only in the mean |
| fig5a-packet-center-enhanced | not reproduced: e1 end above centre in all 4 dishes, not significant |
| fig5b-rotation-center-peak | reproduced, assumed (LOESS centre; unpaired test for within-dish comparisons) |
| fig5c-normalized-contrast | reproduced, assumed (LOESS centre; periphery as both ends) |
| fig3-field-simulation | blocked, data_missing |
| four Fig. 4 claims | blocked, data_missing |
| eleven Fig. 6 to 9 claims | blocked, other: data in DANDI:000875, 001025, 001027, 001028 |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's ROI measurements were made in MetaMorph and are not deposited; here the electrodes are found automatically and ROIs are placed by the layout of Methods 2.4, with a square ROI one step wide (the paper gives no ROI size). With n = 4 or 5 dishes and one 65-um ROI at the centre, the tests on the centre ROI are close to p = 0.05 and change with the ROI width; the LOESS readings are steadier. e1 is not labelled in the files; the electrode brightest after non-rotated treatment is the top-right one in all 8 dishes, which is also the mirror image of Fig. 3's top-left electrode. The background of reading O4 (b) is the median inside the electrode footprints; it changes no outcome. Distances in mm assume 0.65 um per pixel, which the files do not state.

What was found about the dataset. It holds only Fig. 5's 18 dishes; the other monolayer experiments are in four other dandisets that cite the paper. The species is given as Bos taurus (NCBITaxon_9913), where the paper used CHO-K1 hamster cells; the subject age is P1D for a cell line; the image has no pixel size (resolution -1) and no unit; the electrodes are not labelled; the file description speaks of a six-well plate where the paper used 35-mm dishes. Every subject id is unique and no two images are near duplicates (highest correlation 0.815), so the files with the same dish number ("dish2" and "dish2-2", "dish11" in two groups) are different dishes, consistent with their session dates.